# Joint Regression (SUR) with Clustered Errors
## Five equations together: is the school calendar associated with the system?

Five log-concentration regressions, one per pollutant, share identical regressors and are estimated as a system for ONE joint Wald test on the school-term coefficients. We compare an unadjusted specification with the preferred specification controlling for monitoring unit, month and day of week. Each joint test uses TWO covariance matrices: conventional independent-error covariance and weekly-clustered covariance. This isolates the consequences of accounting for temporal dependence. Additional checks include artificial-calendar falsification, leave-one-unit-out analysis and PC1.

**Data access.** The SIMA measurements used here are not
redistributed in this repository (SIMA provided them for this study on the condition
that they are not disseminated); the repository holds the code, the official school
calendar and all derived results. This notebook reads files that notebooks 00 and 01
build from those measurements, and its configuration cell stops with a clear message
if they are missing. To rebuild them, request the annual workbooks from SIMA, place
them in `data_raw/` and run notebook 00 (its header gives the exact file format).

**Role in the analysis.** Day-level contrasts (Eq. 3 of the article): the between-day comparison reported alongside the primary analysis, including the conventional-vs-clustered covariance comparison. The analytical hierarchy of the whole repository is stated in the README.

## 1. Initial setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
import json
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']

output_dir = Path('../data_processed')
resultados_dir = output_dir / 'resultados_metodos'
graficos_dir = output_dir / 'graficos_metodos'

print('Libraries imported successfully')

# Check required inputs (restricted data; see heading)
# Input-data check (restricted data; see heading)
archivos_requeridos = [output_dir / 'panel_pico_hibrido.csv',
                       output_dir / 'panel_pico_MIMA.csv']
faltantes = [str(a) for a in archivos_requeridos if not a.exists()]
if faltantes:
    raise FileNotFoundError(
        ''
        ''
        "Files derived from the SIMA measurements are missing; they are not redistributed in this repository. "
        "Request the annual workbooks from SIMA, place them in data_raw/ and run notebooks 00 and 01. "
        f"Missing: {faltantes}")
print(f"Input data found: {len(archivos_requeridos)} files")

Libraries imported successfully
Input data found: 2 files


## 2. Estimator functions, matching notebook 05

### The mathematics: the same estimator as notebook 05, now applied to LEVELS

The outcome is the log arrival-window concentration rather than the gap, with one equation per pollutant:

$$ \log y^{(j)}_{st}=\alpha^{(j)}+\beta^{(j)}\text{School}_t+\text{controls}+\varepsilon^{(j)}_{st},\qquad j=1,\ldots,5. $$

All five equations share the same matrix $X$, so SUR coefficients coincide with equation-by-equation OLS. The weekly-clustered joint covariance and Wald statistic $W=b'V_b^{-1}b\sim\chi^2_5$ are the same as in notebook 05. The difference is the QUESTION: here we compare arrival-window LEVELS between dates; notebook 05 asks whether the gap WITHIN the day widens during school terms.

### Two covariance matrices for the same hypothesis

- **Conventional:** $\hat V^{conv}=\hat\Sigma\otimes(X'X)^{-1}$, where $\hat\Sigma=\hat U'\hat U/(n-k)$. This assumes independent errors across observations: Monday's error is unrelated to Tuesday's, and the six zones on the same day are unrelated.
- **ISO-week clustered:** allows arbitrary error correlation within a week, including consecutive days and different zones on the same day.

The hypothesis restricts one coefficient per equation, a rank-one hypothesis. With conventional covariance, the Wald statistic corresponds to Hotelling's $T^2$, and the four classical MANOVA statistics—Pillai, Wilks, Hotelling–Lawley and Roy—lead to the same F test. The conventional-versus-clustered comparison therefore keeps the model and tested coefficients fixed while changing the dependence assumption. Below, the conventional result is checked numerically against statsmodels' Pillai trace.

### Assumptions

Here interpretation relies on CONTROLS rather than within-day subtraction: after conditioning on monitoring unit, month and day of week, school-term status must be unrelated to other drivers of pollution for a school-specific interpretation. This is a stronger assumption than the gap design requires. Falsification and leave-one-unit-out checks in Section 6 provide diagnostics, while clustering accounts for dependence within weeks. See notebook 05 for DW diagnostics and notebook 08 for further inference.

### Hypotheses

- Per pollutant: $H_0:\beta_j=0$ against $H_1:\beta_j\ne0$ for arrival-window concentration levels.
- Joint Wald test: $\beta_{CO}=\beta_{NO_2}=\beta_{O_3}=\beta_{PM_{10}}=\beta_{PM_{2.5}}=0$, tested with both covariance estimators.
- PC1: no regime difference in the common pollutant index.

In [2]:
def matriz_diseno(df, especificacion='preferida'):
    "\n    Build the design matrix: intercept, clase and fixed effects.\n    'cruda' = clase only; 'preferida' = zone, month and weekday.\n    "
    piezas = []
    if especificacion != 'cruda':
        piezas.append(pd.get_dummies(df['estacion'], prefix='est', drop_first=True).astype(float))
    if especificacion == 'preferida':
        piezas.append(pd.get_dummies(df['mes'], prefix='mes', drop_first=True).astype(float))
        piezas.append(pd.get_dummies(df['dow'], prefix='dow', drop_first=True).astype(float))
    X = pd.concat([pd.Series(1.0, index=df.index, name='const'),
                   df['clase'].astype(float)] + piezas, axis=1)
    return X

def sur_wald_cluster(df, X, columnas_y, cluster):
    '\n    Equation-wise OLS (identical regressors) and joint clustered covariance -> joint Wald test.\n    '
    Xv = X.values
    n, k = Xv.shape
    XtX_inv = np.linalg.inv(Xv.T @ Xv)
    Y = df[columnas_y].values
    B = XtX_inv @ Xv.T @ Y
    U = Y - Xv @ B
    J = len(columnas_y)
    grupos = df[cluster].values
    orden = np.argsort(grupos, kind='stable')
    Xg, Ug, gg = Xv[orden], U[orden], grupos[orden]
    S = np.zeros((J * k, J * k))
    inicios = np.flatnonzero(np.r_[True, gg[1:] != gg[:-1]])
    fines = np.r_[inicios[1:], len(gg)]
    G = len(inicios)
    for s, e in zip(inicios, fines):
        sg = (Xg[s:e].T @ Ug[s:e]).T.reshape(-1)
        S += np.outer(sg, sg)
    A = np.kron(np.eye(J), XtX_inv)
    V = A @ S @ A
    V *= (G / (G - 1)) * ((n - 1) / (n - k))
    idx_clase = X.columns.get_loc('clase')
    seleccion = [j * k + idx_clase for j in range(J)]
    b = B[idx_clase, :]
    Vb = V[np.ix_(seleccion, seleccion)]
    W = float(b @ np.linalg.inv(Vb) @ b)
    p_chi2 = float(1 - stats.chi2.cdf(W, J))
    resultados_por_var = {}
    for j, col in enumerate(columnas_y):
        se = float(np.sqrt(Vb[j, j]))
        z = b[j] / se
        resultados_por_var[col] = {
            'beta': round(float(b[j]), 4), 'se': round(se, 4),
            'pct': round((np.exp(b[j]) - 1) * 100, 2),
            'ci_lo_pct': round((np.exp(b[j] - 1.96 * se) - 1) * 100, 2),
            'ci_hi_pct': round((np.exp(b[j] + 1.96 * se) - 1) * 100, 2),
            'p': round(2 * (1 - stats.norm.cdf(abs(z))), 6)}
    return {'por_var': resultados_por_var, 'wald_chi2': round(W, 2),
            'p_chi2': round(p_chi2, 6), 'G_clusters': int(G), 'n': int(n), 'k': int(k)}

def sur_wald_convencional(df, X, columnas_y):
    "\n    Apply the same joint test with conventional covariance Sigma (x) (X'X)^-1,\n    assuming independent observations. Return Wald chi2 and, as a cross-check,\n    the statsmodels Pillai trace for the same hypothesis.\n    "
    from statsmodels.multivariate.manova import MANOVA
    Xv = X.values
    n, k = Xv.shape
    XtX_inv = np.linalg.inv(Xv.T @ Xv)
    Y = df[columnas_y].values
    B = XtX_inv @ Xv.T @ Y
    U = Y - Xv @ B
    J = len(columnas_y)
    Sigma = (U.T @ U) / (n - k)
    idx_clase = X.columns.get_loc('clase')
    b = B[idx_clase, :]
    Vb = Sigma * XtX_inv[idx_clase, idx_clase]
    W = float(b @ np.linalg.inv(Vb) @ b)
    p_chi2 = float(stats.chi2.sf(W, J))
    # Pillai test L B = 0, where L selects the 'clase' coefficient
    L = np.zeros((1, k))
    L[0, idx_clase] = 1.0
    prueba = MANOVA(Y, Xv).mv_test(hypotheses=[('clase', L, None)])
    tabla = prueba.results['clase']['stat']
    p_pillai = float(tabla.loc["Pillai's trace", 'Pr > F'])
    F_pillai = float(tabla.loc["Pillai's trace", 'F Value'])
    return {'wald_chi2': round(W, 2), 'p_chi2': float(f"{p_chi2:.3e}"),
            'pillai_F': round(F_pillai, 3), 'pillai_p': float(f"{p_pillai:.3e}"), 'n': int(n), 'k': int(k)}

print('Estimator functions ready')

Estimator functions ready


## 3. Checking the estimator against statsmodels

System coefficients must match equation-by-equation OLS.

In [3]:
panel = pd.read_csv('../data_processed/panel_pico_hibrido.csv', parse_dates=['date'])
columnas_log = [f'l_{c}' for c in contaminantes]

X_pref = matriz_diseno(panel, 'preferida')
resultado_propio = sur_wald_cluster(panel, X_pref, columnas_log, 'wk')

print("=" * 70)
print("CHECK AGAINST STATSMODELS ('clase' coefficient, preferred specification)")
print("=" * 70)
for c in contaminantes:
    modelo_sm = sm.OLS(panel[f'l_{c}'], X_pref).fit()
    beta_sm = modelo_sm.params['clase']
    beta_propio = resultado_propio['por_var'][f'l_{c}']['beta']
    print(f"  {c}: manual = {beta_propio:.6f} | statsmodels = {beta_sm:.6f} | "
          f"coinciden: {abs(beta_propio - beta_sm) < 1e-4}")

CHECK AGAINST STATSMODELS ('clase' coefficient, preferred specification)
  CO: manual = 0.069200 | statsmodels = 0.069187 | coinciden: True
  NO2: manual = 0.093700 | statsmodels = 0.093704 | coinciden: True
  O3: manual = -0.080200 | statsmodels = -0.080232 | coinciden: True
  PM10: manual = 0.056200 | statsmodels = 0.056207 | coinciden: True
  PM2.5: manual = 0.012800 | statsmodels = 0.012848 | coinciden: True


## 4. Both specifications under both imputations

In [4]:
resultados = {}
for nombre_dataset in ['hibrido', 'MIMA']:
    panel_ds = pd.read_csv(f'../data_processed/panel_pico_{nombre_dataset}.csv', parse_dates=['date'])
    panel_ds['mes_est'] = (panel_ds['anio'].astype(str) + '-' + panel_ds['mes'].astype(str)
                           + '-' + panel_ds['estacion'])
    salida = {}
    for especificacion in ['cruda', 'preferida']:
        X = matriz_diseno(panel_ds, especificacion)
        salida[especificacion] = sur_wald_cluster(panel_ds, X, columnas_log, 'wk')
        salida[especificacion]['wald_convencional'] = sur_wald_convencional(panel_ds, X, columnas_log)
    # Alternative clustering (month x unit) and PC1 for the preferred specification
    X = matriz_diseno(panel_ds, 'preferida')
    salida['preferida_cluster_mes_estacion'] = sur_wald_cluster(panel_ds, X, columnas_log, 'mes_est')
    salida['PC1_preferida'] = sur_wald_cluster(panel_ds, X, ['PC1'], 'wk')
    resultados[nombre_dataset] = salida

    print("=" * 70)
    print(f"SUR RESULTS — dataset {nombre_dataset}")
    print("=" * 70)
    for especificacion in ['cruda', 'preferida']:
        r = salida[especificacion]
        print(f"\n[{especificacion}] joint p = {r['p_chi2']} (chi2 = {r['wald_chi2']}, "
              f"weekly-clustered covariance, G = {r['G_clusters']})")
        for c in contaminantes:
            e = r['por_var'][f'l_{c}']
            print(f"  {c}: {e['pct']:+.1f}% [{e['ci_lo_pct']}, {e['ci_hi_pct']}] p = {e['p']}")
    pc1 = salida['PC1_preferida']['por_var']['PC1']
    print(f"\nPC1 (preferred): beta = {pc1['beta']} (p = {pc1['p']})")
    print(f"Alternative month x unit clustering (preferred): "
          f"joint p = {salida['preferida_cluster_mes_estacion']['p_chi2']}")

    print('\nSame joint test with two covariance estimators (impact of ignoring dependence):')
    print(f"  {'specification':<12} {'conventional p':>16} {'weekly-clustered p':>21} {'Pillai p (statsmodels)':>24}")
    for especificacion in ['cruda', 'preferida']:
        r = salida[especificacion]
        cv = r['wald_convencional']
        print(f"  {especificacion:<12} {cv['p_chi2']:>16.2e} {r['p_chi2']:>21.3f} {cv['pillai_p']:>24.2e}")

with open(resultados_dir / 'sur_conjunto.json', 'w') as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)
print(f"\nSaved to: {resultados_dir / 'sur_conjunto.json'}")

SUR RESULTS — dataset hibrido

[cruda] joint p = 0.025265 (chi2 = 12.81, weekly-clustered covariance, G = 158)
  CO: +16.1% [0.43, 34.16] p = 0.043528
  NO2: +17.8% [6.12, 30.73] p = 0.002098
  O3: +2.2% [-8.36, 13.96] p = 0.696733
  PM10: +6.8% [-4.33, 19.13] p = 0.242695
  PM2.5: +10.4% [-0.87, 22.94] p = 0.071821

[preferida] joint p = 0.305423 (chi2 = 6.01, weekly-clustered covariance, G = 158)
  CO: +7.2% [-4.56, 20.33] p = 0.241878
  NO2: +9.8% [0.03, 20.57] p = 0.049223
  O3: -7.7% [-19.62, 5.96] p = 0.254987
  PM10: +5.8% [-7.16, 20.53] p = 0.398658
  PM2.5: +1.3% [-8.84, 12.56] p = 0.811243

PC1 (preferred): beta = 0.2871 (p = 0.04064)
Alternative month x unit clustering (preferred): joint p = 0.001341

Same joint test with two covariance estimators (impact of ignoring dependence):
  specification   conventional p    weekly-clustered p   Pillai p (statsmodels)
  cruda                2.86e-42                 0.025                 1.38e-41
  preferida            9.94e-10        

## 5. Main plot: the two specifications side by side

The plot compares direct arrival-window concentrations, not 24-hour averages. With weekly-clustered covariance, the unadjusted joint comparison rejects at p=0.025, while the preferred adjusted comparison does not reject at p=0.31. School-term and vacation dates have different seasonal compositions, so the distinction between unadjusted and adjusted inference matters.

In [5]:
fig, axes = plt.subplots(1, 5, figsize=(17, 5.5))
especificaciones_grafica = ['cruda', 'preferida']
etiquetas_espec = ['Unadjusted\n(no controls)', 'Preferred\n(zone + month\n+ weekday)']

for i, c in enumerate(contaminantes):
    ax = axes[i]
    r_h = resultados['hibrido']
    valores = [r_h[e]['por_var'][f'l_{c}']['pct'] for e in especificaciones_grafica]
    err_lo = [valores[j] - r_h[e]['por_var'][f'l_{c}']['ci_lo_pct'] for j, e in enumerate(especificaciones_grafica)]
    err_hi = [r_h[e]['por_var'][f'l_{c}']['ci_hi_pct'] - valores[j] for j, e in enumerate(especificaciones_grafica)]
    colores_barras = ['#95a5a6', '#3498db']
    bars = ax.bar(range(len(especificaciones_grafica)), valores, yerr=[err_lo, err_hi], color=colores_barras,
                  alpha=0.85, edgecolor='black', capsize=4)
    for bar, valor in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + (0.8 if valor >= 0 else -3.0),
                f'{valor:+.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(c, fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(especificaciones_grafica)))
    ax.set_xticklabels(etiquetas_espec, fontsize=8.5)
    if i == 0:
        ax.set_ylabel('School-term effect (%)', fontsize=11, fontweight='bold')

fig.suptitle('School-term association with arrival-window levels\n(percent effect with transformed 95% CI; weekly-clustered errors; hybrid imputation)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(graficos_dir / '02_sur_especificaciones.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {graficos_dir / '02_sur_especificaciones.png'}")

Plot saved to: ../data_processed/graficos_metodos/02_sur_especificaciones.png


## 6. Sensitivity: artificial-calendar falsification and leave-one-unit-out analysis

The artificial indicator alternates even and odd ISO weeks within one regime, either vacation only or school term only. It does not represent an actual change in school attendance. Consecutive weeks can differ in autocorrelation, holidays and other activity patterns, so a null result provides limited support rather than proof of validity.

In [6]:
panel['wknum'] = (panel['date'].dt.isocalendar().year.astype(int) * 100
                  + panel['date'].dt.isocalendar().week.astype(int))

robustez = {}
for nombre, sub in (('placebo_vacaciones', panel[panel['clase'] == 0]),
                    ('placebo_clases', panel[panel['clase'] == 1])):
    s = sub.copy()
    s['clase'] = (s['wknum'] % 2).astype(int)
    Xs = matriz_diseno(s, 'preferida')
    robustez[nombre] = sur_wald_cluster(s, Xs, columnas_log, 'wk')

robustez['loso'] = {}
for estacion_m in sorted(panel['estacion'].unique()):
    s = panel[panel['estacion'] != estacion_m]
    Xs = matriz_diseno(s, 'preferida')
    r = sur_wald_cluster(s, Xs, columnas_log, 'wk')
    robustez['loso'][f'sin_{estacion_m}'] = {
        'p_conjunto': r['p_chi2'],
        'CO_pct': r['por_var']['l_CO']['pct'],
        'NO2_pct': r['por_var']['l_NO2']['pct']}

print("=" * 70)
print("ROBUSTNESS")
print("=" * 70)
print(f"Artificial indicator within vacation (week parity): p = {robustez['placebo_vacaciones']['p_chi2']}")
print(f"Artificial indicator within school term: p = {robustez['placebo_clases']['p_chi2']}")
print('\nLeave-one-unit-out (joint p after excluding each unit):')
for nombre, r in robustez['loso'].items():
    print(f"  {nombre}: p = {r['p_conjunto']} (CO {r['CO_pct']:+.1f}%, NO2 {r['NO2_pct']:+.1f}%)")

with open(resultados_dir / 'sur_robustez.json', 'w') as f:
    json.dump(robustez, f, indent=2, ensure_ascii=False)
print(f"\nSaved to: {resultados_dir / 'sur_robustez.json'}")

print('\nCOMPLETED')

ROBUSTNESS
Artificial indicator within vacation (week parity): p = 0.5983
Artificial indicator within school term: p = 0.758018

Leave-one-unit-out (joint p after excluding each unit):
  sin_CENTRO: p = 0.283671 (CO +7.5%, NO2 +9.8%)
  sin_NORESTE3: p = 0.314294 (CO +6.8%, NO2 +10.3%)
  sin_NOROESTE3: p = 0.197889 (CO +9.1%, NO2 +9.7%)
  sin_NORTE2: p = 0.357353 (CO +6.7%, NO2 +10.2%)
  sin_SUR: p = 0.356457 (CO +6.8%, NO2 +8.9%)
  sin_SUROESTE2: p = 0.384413 (CO +6.2%, NO2 +10.0%)

Saved to: ../data_processed/resultados_metodos/sur_robustez.json

COMPLETED
